## 0.11. Veri Kümesini Yükleme

Modeli 1100 Türkçe isim üzerinde eğiteceğiz. Her isim kısa bir belge
(document) olarak kabul edilecektir.

Veri kümesindeki isimler önce temizlenecek, küçük harfe dönüştürülecek
ve ardından eğitim sırasının etkisini azaltmak için karıştırılacaktır.

In [1]:
import random
import math
docs = [
    line.strip()
        .replace(" ", "")## aradaki boşlukları da sildim.
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
    for line in open("../data/raw_names.txt", encoding="utf-8")
        .read()
        .strip()
        .split("\n")
    if line.strip()
]

random.shuffle(docs)

In [2]:
# Listeyi bulunduğu yerde karıştırır
random.shuffle(docs)

print("Karıştırılmış ilk 10 isim:")
print(docs[:10])

Karıştırılmış ilk 10 isim:
['şerife', 'erdoğan', 'malik', 'neriman', 'gülşen', 'timur', 'huri', 'hanifi', 'eslem', 'fuat']


## 0.2. Tokenizer

Dil modeli karakterlerle doğrudan çalışamaz. Bu nedenle her farklı
karaktere sayısal bir kimlik, yani token ID verilecektir.

Bu projede karakter düzeyinde tokenization kullanılacaktır.

In [3]:
uchars = sorted(set(''.join(docs)))

BOS = len(uchars)
vocab_size = len(uchars) + 1

print(uchars)
print("Karakter sayısı:", len(uchars))
print("BOS token ID:", BOS)
print("Vocabulary boyutu:", vocab_size)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y', 'z', 'ç', 'ö', 'ü', 'ğ', 'ı', 'ş']
Karakter sayısı: 29
BOS token ID: 29
Vocabulary boyutu: 30


## 0.3. Sayım Tablosu

Bigram modelimiz iki boyutlu bir sayım tablosudur.

- Satırlar mevcut tokenı temsil eder.
- Sütunlar bir sonraki tokenı temsil eder.
- Her hücre, bir tokenın ardından diğer tokenın kaç kez geldiğini tutar.

Başlangıçta bütün hücreler sıfırdır.

In [4]:
state_dict = [[0] * vocab_size for _ in range(vocab_size)]
print("Toplam hücre sayısı:", len(state_dict) * len(state_dict[0]))


Toplam hücre sayısı: 900


## 0.4. Bigram Modeli

Bigram modeli, mevcut tokenı kullanarak bir sonraki tokenın olasılıklarını
hesaplar.

Model yalnızca iki token arasındaki ilişkiye bakar:

- Mevcut token
- Bir sonraki token

Daha önce gelen tokenları hatırlamaz.

Laplace smoothing
Fonksiyonda iki ekleme bulunuyor.Böylece hiç görülmeyen bir geçişin olasılığı çok küçük olur ama tamamen sıfır olmaz.
Her hücreye 1 eklenmiş gibi davranıyoruz. Bizde bir satırda 30 hücre olduğu için toplam sayıya da 30 eklemeliyiz

In [5]:
def bigram(token_id):
    row = state_dict[token_id]
    total = sum(row) + vocab_size
    return [(c + 1) / total for c in row]

## 0.5. Modelin Eğitilmesi

Her eğitim adımında bir isim seçilir, tokenlara dönüştürülür ve başına
ve sonuna BOS tokenı eklenir.

Model önce ismi ne kadar iyi tahmin ettiğini loss ile ölçer. Ardından
isimdeki karakter geçişlerinin sayılarını bir artırır.

In [6]:
num_steps =1000
for step in range(num_steps):
    doc =docs[step% len(docs)]
    tokens =[BOS] + [uchars.index(ch)for ch in doc]+[BOS]
    n=len(tokens)-1

    losses =[]
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id],tokens[pos_id+1]
        probs =bigram(token_id)
        loss_t= -math.log(probs[target_id])
        losses.append(loss_t)
    loss = (1/n)*sum(losses)

    for pos_id in range(n):
        token_id,target_id =tokens[pos_id],tokens[pos_id+1]
        state_dict[token_id][target_id]+=1
        

In [7]:
print("Son işlenen isim:", doc)
print("Son loss:", loss)
print("Tablodaki toplam geçiş sayısı:", sum(sum(row) for row in state_dict))

Beklenen geçiş: 6674
Tablodaki geçiş: 6674
Eşleşiyor mu? True
